In [1]:
# @title Imports and Notebook Utilities
# This block is mostly taken from the self-org textures notebook.
import os
import io
import PIL.Image, PIL.ImageDraw
import base64
import zipfile
import json
import requests
import numpy as np
import matplotlib.pylab as pl
import glob

from IPython.display import Image, HTML, Markdown, clear_output
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

os.environ['FFMPEG_BINARY'] = 'ffmpeg'
import moviepy.editor as mvp
from moviepy.video.io.ffmpeg_writer import FFMPEG_VideoWriter


def imread(url, max_size=None, mode=None):
    if isinstance(url, str) and url.startswith(('http:', 'https:')):
        # wikimedia requires a user agent
        headers = {
            "User-Agent": "Requests in Colab/0.0 (https://colab.research.google.com/; no-reply@google.com) requests/0.0"
        }
        r = requests.get(url, headers=headers)
        f = io.BytesIO(r.content)
    else:
        f = url
    img = PIL.Image.open(f)
    if max_size is not None:
        img.thumbnail((max_size, max_size), PIL.Image.LANCZOS)
    if mode is not None:
        img = img.convert(mode)
    img = np.float32(img) / 255.0
    return img


def np2pil(a):
    if a.dtype in [np.float32, np.float64]:
        a = np.uint8(np.clip(a, 0, 1) * 255)
    return PIL.Image.fromarray(a)


def imwrite(f, a, fmt=None):
    a = np.asarray(a)
    if isinstance(f, str):
        fmt = f.rsplit('.', 1)[-1].lower()
        if fmt == 'jpg':
            fmt = 'jpeg'
        f = open(f, 'wb')
    np2pil(a).save(f, fmt, quality=95)


def imencode(a, fmt='jpeg'):
    a = np.asarray(a)
    if len(a.shape) == 3 and a.shape[-1] == 4:
        fmt = 'png'
    f = io.BytesIO()
    imwrite(f, a, fmt)
    return f.getvalue()


def im2url(a, fmt='jpeg'):
    encoded = imencode(a, fmt)
    base64_byte_string = base64.b64encode(encoded).decode('ascii')
    return 'data:image/' + fmt.upper() + ';base64,' + base64_byte_string


def imshow(a, fmt='jpeg', id=None):
    return display(Image(data=imencode(a, fmt)), display_id=id)


def grab_plot(close=True):
    """Return the current Matplotlib figure as an image"""
    fig = pl.gcf()
    fig.canvas.draw()
    img = np.array(fig.canvas.renderer._renderer)
    a = np.float32(img[..., 3:] / 255.0)
    img = np.uint8(255 * (1.0 - a) + img[..., :3] * a)  # alpha
    if close:
        pl.close()
    return img



def zoom(img, scale=4):
    img = np.repeat(img, scale, 0)
    img = np.repeat(img, scale, 1)
    return img


class VideoWriter:
    def __init__(self, filename='_autoplay.mp4', fps=30.0, **kw):
        self.writer = None
        self.params = dict(filename=filename, fps=fps, **kw)

    def add(self, img):
        img = np.asarray(img)
        if self.writer is None:
            h, w = img.shape[:2]
            self.writer = FFMPEG_VideoWriter(size=(w, h), **self.params)
        if img.dtype in [np.float32, np.float64]:
            img = np.uint8(img.clip(0, 1) * 255)
        if len(img.shape) == 2:
            img = np.repeat(img[..., None], 3, -1)
        self.writer.write_frame(img)

    def close(self):
        if self.writer:
            self.writer.close()

    def __enter__(self):
        return self

    def __exit__(self, *kw):
        self.close()
        if self.params['filename'] == '_autoplay.mp4':
            self.show()

    def show(self, **kw):
        self.close()
        fn = self.params['filename']
        display(mvp.ipython_display(fn, **kw))

!nvidia-smi -L

GPU 0: NVIDIA L4 (UUID: GPU-5a006382-6584-7e3e-88cf-0c14bd72b7ed)


In [2]:
import torch
import torchvision.models as models

torch.set_default_tensor_type('torch.cuda.FloatTensor')

In [ ]:
#@title VGG16 Sliced OT Style Loss
import torch.nn.functional as F

def calc_styles_vgg(imgs, vgg):
    style_layers = [1, 6, 11, 18, 25]
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    x = (imgs - mean) / std
    b, c, h, w = x.shape
    features = [x.reshape(b, c, h * w)]
    for i, layer in enumerate(vgg[:max(style_layers) + 1]):
        x = layer(x)
        if i in style_layers:
            b, c, h, w = x.shape
            features.append(x.reshape(b, c, h * w))
    return features

def project_sort(x, proj):
    return torch.einsum('bcn,cp->bpn', x, proj).sort()[0]

def ot_loss(source, target, proj_n=32):
    ch, n = source.shape[-2:]
    projs = F.normalize(torch.randn(ch, proj_n), dim=0)
    source_proj = project_sort(source, projs)
    target_proj = project_sort(target, projs)
    target_interp = F.interpolate(target_proj, n, mode='nearest')
    return (source_proj - target_interp).square().sum()

def create_vgg_loss(vgg, target_img):
    yy = calc_styles_vgg(target_img, vgg)
    def loss_f(imgs):
        xx = calc_styles_vgg(imgs, vgg)
        return sum(ot_loss(x, y) for x, y in zip(xx, yy))
    return loss_f


In [ ]:
#@title Load VGG and Target image {vertical-output: true}
vgg = models.vgg16(weights='IMAGENET1K_V1').features

from google.colab import files

print("Please upload your style image:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f'Using uploaded file: "{filename}"')

style_img = imread(io.BytesIO(uploaded[filename]), max_size=128)
style_img_torch = torch.tensor(style_img).permute(2, 0, 1).unsqueeze(0)

with torch.no_grad():
  loss_fn = create_vgg_loss(vgg, style_img_torch)
imshow(style_img)

In [ ]:
#@title ReactionDiffusionCA Architecture
from scipy.ndimage import gaussian_filter

def laplacian(x):
    """Apply Laplacian filter to all channels independently."""
    b, ch, h, w = x.shape
    # Unnormalized Laplacian for consistency with NoiseNCA
    lap = torch.tensor([[1.0, 2.0, 1.0],
                        [2.0, -12.0, 2.0],
                        [1.0, 2.0, 1.0]])

    # Depthwise convolution with circular padding
    y = x.reshape(b * ch, 1, h, w)
    y = torch.nn.functional.pad(y, [1, 1, 1, 1], "circular")
    y = torch.nn.functional.conv2d(y, lap[None, None])
    return y.reshape(b, ch, h, w)


class ReactionDiffusionCA(torch.nn.Module):
    def __init__(self, chn=12, hidden_n=128, noise_level=0.1):
        super().__init__()
        self.chn = chn
        self.register_buffer("noise_level", torch.tensor([noise_level]))

        # Reaction network (operates on state directly)
        self.w1 = torch.nn.Conv2d(chn, hidden_n, 1, bias=True)
        self.w2 = torch.nn.Conv2d(hidden_n, chn, 1, bias=False)

        # Initialize weights
        torch.nn.init.xavier_normal_(self.w1.weight, gain=0.2)
        torch.nn.init.zeros_(self.w2.weight)

        # Per-channel diffusion coefficients (multi-scale)
        # Repeat pattern [0.125, 0.25, 0.5, 1.0] across channels
        n_groups = chn // 4
        diff_coef = torch.tensor([0.125, 0.25, 0.5, 1.0]).repeat(n_groups)
        # Handle remainder channels
        if chn % 4 != 0:
            diff_coef = torch.cat([diff_coef, torch.ones(chn % 4)])
        self.register_buffer("diff_coef", diff_coef)

    def forward(self, x, r=1.0, d=1.0, noise=None):
        """
        Args:
            x: State tensor [b, chn, h, w]
            r: Reaction rate scaling (default 1.0)
            d: Diffusion rate scaling (default 1.0)
            noise: Noise level to add (default None)
        """
        # Add noise if specified
        if noise is not None:
            x = x + torch.randn_like(x) * noise

        # DIFFUSION TERM: Laplacian with per-channel coefficients
        diff = laplacian(x) * self.diff_coef[None, :, None, None]

        # REACTION TERM: Learned nonlinear dynamics with ReLU activation
        y = self.w1(x)
        y = torch.relu(y)  # ReLU activation (same as NoiseNCA)
        react = self.w2(y)

        # Explicit reaction-diffusion update
        x = x + diff * d + react * r
        return x

    def seed(self, n, h=128, w=128, seed_type='uniform'):
        """Generate initial state.

        Args:
            n: Batch size
            h, w: Image dimensions
            seed_type: 'uniform' (default NCA-style) or 'gaussian_blobs' (RD-style)
        """
        if seed_type == 'gaussian_blobs':
            return self.seed_gaussian_blobs(n, h, w)
        else:
            # Default: uniform random noise
            return (torch.rand(n, self.chn, h, w) - 0.5) * self.noise_level

    def seed_gaussian_blobs(self, n, h=128, w=128, spot_prob=0.005, spread=3.0):
        """Create seed states with scattered gaussian blobs (from RD paper).

        This creates sparse random spots and blurs them with a Gaussian filter,
        creating smooth blob-like initial conditions. Only RGB channels are
        initialized; hidden channels start at 0.

        Args:
            n: Batch size
            h, w: Image dimensions
            spot_prob: Probability of a spot at each pixel (default 0.005)
            spread: Gaussian blur sigma (default 3.0)
        """
        # Create sparse random spots (only 0.5% of pixels are 1.0)
        x = np.floor(np.random.uniform(0, 1, (n, h, w, 1)) + spot_prob)

        # Blur with Gaussian filter (mode='wrap' for toroidal boundary)
        x = gaussian_filter(x, sigma=[0.0, spread, spread, 0.0], mode='wrap')

        # Scale by spread^2 to compensate for blur normalization
        x = x * spread ** 2

        # Replicate to RGB channels (3 channels)
        x = np.repeat(x, 3, axis=-1)

        # Pad with zeros for remaining hidden channels
        x = np.pad(x, [(0, 0), (0, 0), (0, 0), (0, self.chn - 3)])

        # Convert to PyTorch tensor and permute to [n, chn, h, w]
        return torch.tensor(x, dtype=torch.float32).permute(0, 3, 1, 2)


def to_rgb(s):
    """Extract RGB channels from state."""
    return s[..., :3, :, :] + 0.5


param_n = sum(p.numel() for p in ReactionDiffusionCA().parameters())
print('ReactionDiffusionCA param count:', param_n)

# Visualize both seed types
print('\nUniform noise seeds (NCA-style):')
img = to_rgb(ReactionDiffusionCA().seed(4, 128, seed_type='uniform'))
imshow(np.hstack(img.permute(0, 2, 3, 1).cpu().numpy()))

print('\nGaussian blob seeds (RD-style):')
img = to_rgb(ReactionDiffusionCA().seed(4, 128, seed_type='gaussian_blobs'))
imshow(np.hstack(img.permute(0, 2, 3, 1).cpu().numpy()))

In [ ]:
#@title Setup Training
import os
from google.colab import files

# Check for checkpoint files in Colab storage
checkpoint_files = glob.glob('*.pt')

if checkpoint_files:
    print(f"Found {len(checkpoint_files)} checkpoint file(s) in Colab storage:")
    for i, f in enumerate(checkpoint_files):
        file_size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  [{i}] {f} ({file_size_mb:.2f} MB)")

    choice = input("\nEnter number to load, 'u' to upload, or Enter for fresh start: ").strip()

    if choice == 'u':
        print("Please upload your checkpoint file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        print(f'Loading checkpoint from uploaded file: "{filename}"')
        checkpoint = torch.load(filename)
    elif choice.isdigit() and 0 <= int(choice) < len(checkpoint_files):
        filename = checkpoint_files[int(choice)]
        print(f'Loading checkpoint from: "{filename}"')
        checkpoint = torch.load(filename)
    else:
        checkpoint = None
else:
    print("No checkpoint files found in Colab storage.")
    upload_choice = input("Upload a checkpoint file? (y/n, default=n): ").strip().lower()

    if upload_choice == 'y':
        print("Please upload your checkpoint file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        print(f'Loading checkpoint from: "{filename}"')
        checkpoint = torch.load(filename)
    else:
        checkpoint = None

# Initialize or load model
if checkpoint is not None:
    model = ReactionDiffusionCA()
    model.load_state_dict(checkpoint['model_state_dict'])
    start_iter = checkpoint.get('iteration', 0)
    print(f'Loaded checkpoint from iteration {start_iter}')
else:
    print('Initializing new ReactionDiffusionCA model...')
    model = ReactionDiffusionCA()
    start_iter = 0

# Optimizer
opt = torch.optim.Adam(model.parameters(), 1e-3, capturable=True)

# Adaptive learning rate scheduler
lr_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt,
    mode='min',
    factor=0.3,
    patience=500,
    threshold=0.01,
    threshold_mode='rel',
    min_lr=1e-6,
    verbose=True
)

# Load optimizer and scheduler state if resuming
if checkpoint is not None:
    if 'optimizer_state_dict' in checkpoint:
        opt.load_state_dict(checkpoint['optimizer_state_dict'])
    if 'scheduler_state_dict' in checkpoint:
        lr_sched.load_state_dict(checkpoint['scheduler_state_dict'])
    if 'loss_log' in checkpoint:
        loss_log = checkpoint['loss_log']
    else:
        loss_log = []
    if 'pool' in checkpoint:
        pool = checkpoint['pool']
    else:
        with torch.no_grad():
            pool = model.seed(256)
else:
    loss_log = []
    with torch.no_grad():
        pool = model.seed(256)

print(f'Starting from iteration {start_iter}')
print(f'Pool shape: {pool.shape}')

In [ ]:
# @title Training loop {vertical-output: true}

# Training parameters
num_iterations = 5000  #@param {type: "integer"}
reaction_rate = 1.0    #@param {type: "number"}
diffusion_rate = 1.0   #@param {type: "number"}
noise_level = 0.05     #@param {type: "number"}
seed_type = "uniform"  #@param ["uniform", "gaussian_blobs"]

for i in range(start_iter, start_iter + num_iterations):
    with torch.no_grad():
        batch_idx = np.random.choice(len(pool), 4, replace=False)
        s = pool[batch_idx]
        if i % 8 == 0:
            s[:1] = model.seed(1, seed_type=seed_type)

    # Random number of steps
    step_n = np.random.randint(32, 96)
    for k in range(step_n):
        s = model(s, r=reaction_rate, d=diffusion_rate, noise=noise_level)

    # Compute loss
    overflow_loss = (s - s.clamp(-1.0, 1.0)).abs().sum()
    loss = loss_fn(to_rgb(s)) + overflow_loss

    # Backward pass and optimize
    with torch.no_grad():
        loss.backward()
        for p in model.parameters():
            p.grad /= (p.grad.norm() + 1e-8)  # Normalize gradients
        opt.step()
        opt.zero_grad()
        lr_sched.step(loss)  # Adaptive scheduler
        pool[batch_idx] = s

        loss_log.append(loss.item())

        # Display progress
        if i % 5 == 0:
            current_lr = opt.param_groups[0]['lr']
            display(Markdown(f'''
        iteration: {i}
        loss: {loss.item():.2e}
        lr: {current_lr:.2e}'''), display_id='stats')

        # Plot and visualize
        if i % 32 == 0:
            pl.plot(loss_log, '.', alpha=0.1)
            pl.yscale('log')
            pl.ylim(np.min(loss_log), loss_log[0])
            pl.tight_layout()
            imshow(grab_plot(), id='log')
            imgs = to_rgb(s).permute([0, 2, 3, 1]).cpu()
            imshow(np.hstack(imgs), id='batch')

        # Save checkpoint every 100 iterations
        if i % 100 == 0 and i > start_iter:
            checkpoint = {
                'iteration': i,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': opt.state_dict(),
                'scheduler_state_dict': lr_sched.state_dict(),
                'loss_log': loss_log,
                'pool': pool,
            }
            torch.save(checkpoint, f'rd_checkpoint_iter_{i}.pt')
            print(f'\nSaved checkpoint at iteration {i}')

In [ ]:
#@title Save Model Weights
import torch
from google.colab import files

# Save weights-only file (for demo)
weights_filename = 'rd_weights.pt'
torch.save(model.state_dict(), weights_filename)
print(f"Model weights saved as '{weights_filename}'")

# Save full checkpoint (for resuming training)
checkpoint_filename = f'rd_checkpoint_final.pt'
checkpoint = {
    'iteration': i,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': opt.state_dict(),
    'scheduler_state_dict': lr_sched.state_dict(),
    'loss_log': loss_log,
    'pool': pool,
}
torch.save(checkpoint, checkpoint_filename)
print(f"Full checkpoint saved as '{checkpoint_filename}'")

# Offer to download
print("\nDownload files:")
files.download(weights_filename)
files.download(checkpoint_filename)